In [1]:
import os
import subprocess
import time

# 1. Install packages
!pip install -q transformers accelerate fastapi uvicorn python-multipart qwen-vl-utils opencv-python-headless torchvision

# 2. Write the API Server script
api_code = """
import io, cv2, torch, os
import numpy as np
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from torchvision import transforms, models
from torch import nn
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

print("Loading ResNet-18 Gatekeeper...")
model_path = "/content/tier1_best.pth"
if not os.path.exists(model_path):
    raise FileNotFoundError("Could not find tier1_best.pth! Did you upload it to the left sidebar?")

resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 1)
# Using weights_only=False to bypass the PyTorch 2.6 security check
resnet.load_state_dict(torch.load(model_path, map_location="cuda", weights_only=False))
resnet = resnet.to("cuda")
resnet.eval()
res_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

print("Loading Qwen2-VL-2B into GPU Memory...")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
vlm = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda")
processor = AutoProcessor.from_pretrained(model_id)
print("Ensemble Server Ready!")

@app.post("/characterize")
async def characterize(crop: UploadFile = File(...), full: UploadFile = File(...)):
    crop_cv = cv2.imdecode(np.frombuffer(await crop.read(), np.uint8), cv2.IMREAD_COLOR)
    full_cv = cv2.imdecode(np.frombuffer(await full.read(), np.uint8), cv2.IMREAD_COLOR)
    crop_pil = Image.fromarray(cv2.cvtColor(crop_cv, cv2.COLOR_BGR2RGB))

    # 1. GATEKEEPER CHECK (ResNet-18)
    with torch.no_grad():
        prob = torch.sigmoid(resnet(res_transform(crop_pil).unsqueeze(0).to("cuda"))).item()

    if prob < 0.50:
        return {
            "stub": False,
            "attributes": {"Status": "REJECTED BY GATEKEEPER", "Malignancy Risk": f"{prob*100:.1f}%", "Model": "ResNet-18"},
            "similar_cases": [],
            "narrative": f"False Positive. The ResNet-18 Gatekeeper analyzed this region and determined it is normal anatomy (Confidence: {100 - prob*100:.1f}% Benign). VLM was not triggered."
        }

    # 2. VLM DESCRIBER
    result = cv2.matchTemplate(full_cv, crop_cv, cv2.TM_CCOEFF_NORMED)
    _, _, _, max_loc = cv2.minMaxLoc(result)
    cv2.rectangle(full_cv, max_loc, (max_loc[0] + crop_cv.shape[1], max_loc[1] + crop_cv.shape[0]), (0, 0, 255), 4)
    marked_img = Image.fromarray(cv2.cvtColor(full_cv, cv2.COLOR_BGR2RGB))

    prompt = "You are a radiologist. Describe the shape, margins, and texture of the distinct lung nodule enclosed in the RED BOX in one sentence."
    messages = [{"role": "user", "content": [{"type": "image", "image": marked_img}, {"type": "text", "text": prompt}]}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[marked_img], padding=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        answer = processor.batch_decode([out[len(inp):] for inp, out in zip(inputs.input_ids, vlm.generate(**inputs, max_new_tokens=80))], skip_special_tokens=True)[0]

    return {
        "stub": False,
        "attributes": {"Status": "VLM Analysis Complete", "Malignancy Risk": f"{prob*100:.1f}%", "Model": "ResNet-18 + Qwen2-VL"},
        "similar_cases": [],
        "narrative": f"[Gatekeeper Approved] {answer}"
    }
"""
with open("tier1_ensemble_service.py", "w") as f:
    f.write(api_code)

# 3. Setup Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!pkill -f uvicorn
!pkill -f cloudflared
time.sleep(1)

print("Opening public Cloudflare Tunnel...")
tunnel = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:9000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

while True:
    line = tunnel.stdout.readline()
    if ".trycloudflare.com" in line:
        url = line[line.find("https://"):line.find(".trycloudflare.com") + 18]
        break

print("\n" + "🔥"*25)
print("🌐 YOUR FINAL ENSEMBLE CLOUD URL:")
print(url)
print("🔥"*25 + "\n")

# 4. Start Server
!python -m uvicorn tier1_ensemble_service:app --host 127.0.0.1 --port 9000

Opening public Cloudflare Tunnel...

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🌐 YOUR FINAL ENSEMBLE CLOUD URL:
https://wrist-voices-surrounding-rpg.trycloudflare.com
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

Loading ResNet-18 Gatekeeper...
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/__main__.py", line 4, in <module>
    uvicorn.main()
    ~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1631, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1552, in main
    rv = self.invoke(ctx)
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1415, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py",